# Fine-tuning de modelo médico com TinyLlama e LoRA

Este notebook tem como objetivo executar o fine-tuning de um modelo base (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) usando um dataset de perguntas e respostas médicas.

O processo inclui:
- verificação de GPU no Google Colab
- instalação de dependências
- carregamento do projeto a partir do GitHub
- importação da função de fine-tuning
- leitura e validação do dataset
- execução de um teste reduzido de treinamento
- salvamento dos artefatos no Google Drive
- teste de inferência com o modelo ajustado

In [17]:
import os
import sys
import platform
import torch
import pandas as pd
import importlib.util
import traceback
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

print("Python:", sys.version)
print("Sistema:", platform.platform())
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU não está ativa. Vá em Ambiente de execução > Alterar tipo de execução > GPU")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Sistema: Linux-6.6.113+-x86_64-with-glibc2.35
CUDA disponível: True
GPU: Tesla T4


## Verificação do ambiente de execução

Esta célula verifica se o ambiente do Google Colab possui GPU habilitada, além de mostrar a versão do Python e informações de hardware.

Essa etapa é importante porque o fine-tuning com modelos de linguagem exige aceleração por GPU para ser viável.

In [18]:
!pip -q install -U \
  transformers \
  datasets \
  peft \
  accelerate \
  bitsandbytes \
  sentencepiece \
  scikit-learn \
  pandas \
  einops

## Instalação de dependências

Nesta etapa são instaladas as bibliotecas necessárias para o treinamento e inferência, incluindo:

- `transformers`: carregamento do modelo e tokenizer
- `datasets`: manipulação de datasets
- `peft`: aplicação de LoRA
- `accelerate`: suporte à execução eficiente em GPU
- `bitsandbytes`: quantização em 4 bits
- `sentencepiece`: suporte a tokenização
- `pandas` e `scikit-learn`: manipulação e divisão dos dados

In [ ]:
%cd /content
!rm -rf repo_fiap
!git clone https://github.com/KeityPires/tech-challenge-ia-diagnostico repo_fiap
%cd /content/repo_fiap
!find /content/repo_fiap -name "fine_tuning.py"

/content
Cloning into 'repo_fiap'...
remote: Enumerating objects: 385, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 385 (delta 71), reused 95 (delta 34), pack-reused 249 (from 1)
Receiving objects: 100% (385/385), 52.11 MiB | 14.63 MiB/s, done.
Resolving deltas: 100% (182/182), done.
Updating files: 100% (39/39), done.
/content/repo_fiap
/content/repo_fiap/src/llm/fine_tuning.py


## Clonagem do projeto

Esta célula clona o repositório do GitHub para dentro do ambiente do Colab.

Isso permite reutilizar o código Python desenvolvido no projeto, especialmente a função `run_medical_finetuning`, sem precisar reescrever toda a lógica no notebook.

In [ ]:
FINE_TUNING_PATH = "/content/repo_fiap/src/llm/fine_tuning.py"

In [ ]:
import os
print("Arquivo existe?", os.path.exists(FINE_TUNING_PATH))
if not os.path.exists(FINE_TUNING_PATH):
    raise FileNotFoundError(FINE_TUNING_PATH)

Arquivo existe? True


In [ ]:
import importlib.util
import sys
import traceback

MODULE_NAME = "fiap_project_fine_tuning"

spec = importlib.util.spec_from_file_location(MODULE_NAME, FINE_TUNING_PATH)
mod = importlib.util.module_from_spec(spec)
sys.modules[MODULE_NAME] = mod

try:
    spec.loader.exec_module(mod)
    print("Import do arquivo OK")
except Exception:
    traceback.print_exc()
    raise

print("Tem run_medical_finetuning?", hasattr(mod, "run_medical_finetuning"))
print("Tem prepare_finetuning_dataset?", hasattr(mod, "prepare_finetuning_dataset"))
print("Tem split_finetuning_dataset?", hasattr(mod, "split_finetuning_dataset"))

Import do arquivo OK
Tem run_medical_finetuning? True
Tem prepare_finetuning_dataset? True
Tem split_finetuning_dataset? True


In [ ]:
run_medical_finetuning = mod.run_medical_finetuning
prepare_finetuning_dataset = mod.prepare_finetuning_dataset
split_finetuning_dataset = mod.split_finetuning_dataset
save_finetuning_dataset = mod.save_finetuning_dataset

print("Funções carregadas com sucesso.")

Funções carregadas com sucesso.


## Importação do módulo de fine-tuning

Nesta etapa, o arquivo `fine_tuning.py` é carregado diretamente por caminho absoluto.

Essa abordagem foi escolhida para evitar conflitos de importação de pacotes no Colab, garantindo que o notebook use exatamente o arquivo do projeto.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Montagem do Google Drive

Aqui o Google Drive é montado para permitir salvar os artefatos gerados no treinamento, como:

- adapter LoRA
- tokenizer
- checkpoints
- configurações do treinamento

Isso evita perda de arquivos quando a sessão do Colab for encerrada.

In [ ]:
import pandas as pd

DATA_PATH = "data/medical_qa/processed/medquad_cancergov_qa_finetuning.jsonl"
df_medquad = pd.read_json(DATA_PATH, lines=True)

print(df_medquad.head())
print(df_medquad.columns.tolist())
print(df_medquad.shape)

                                            question  \
0  What is (are) Adult Acute Lymphoblastic Leukem...   
1  What are the symptoms of Adult Acute Lymphobla...   
2  How to diagnose Adult Acute Lymphoblastic Leuk...   
3  What is the outlook for Adult Acute Lymphoblas...   
4  Who is at risk for Adult Acute Lymphoblastic L...   

                                              answer  \
0  Key Points\n                    - Adult acute ...   
1  Signs and symptoms of adult ALL include fever,...   
2  Tests that examine the blood and bone marrow a...   
3  Certain factors affect prognosis (chance of re...   
4  Previous chemotherapy and exposure to radiatio...   

                                                text  
0  ### Instrução:\nResponda à pergunta médica de ...  
1  ### Instrução:\nResponda à pergunta médica de ...  
2  ### Instrução:\nResponda à pergunta médica de ...  
3  ### Instrução:\nResponda à pergunta médica de ...  
4  ### Instrução:\nResponda à pergunta médica de ..

## Carregamento do dataset

Nesta célula, o dataset de fine-tuning é carregado a partir de um arquivo `.jsonl`.

O dataset contém pares de pergunta e resposta médicas e será usado como base para o ajuste fino do modelo.

As colunas esperadas pelo código são:
- `question`
- `answer`

In [ ]:
required = {"question", "answer"}
missing = required - set(df_medquad.columns)
print("Colunas faltando:", missing)

if missing:
    raise ValueError(f"Seu dataset precisa ter as colunas {required}, mas faltam {missing}")

print("Nulos:")
print(df_medquad[["question", "answer"]].isna().sum())

print("Amostra limpa:")
print(df_medquad[["question", "answer"]].dropna().head(3))

Colunas faltando: set()
Nulos:
question    0
answer      0
dtype: int64
Amostra limpa:
                                            question  \
0  What is (are) Adult Acute Lymphoblastic Leukem...   
1  What are the symptoms of Adult Acute Lymphobla...   
2  How to diagnose Adult Acute Lymphoblastic Leuk...   

                                              answer  
0  Key Points\n                    - Adult acute ...  
1  Signs and symptoms of adult ALL include fever,...  
2  Tests that examine the blood and bone marrow a...  


## Validação dos dados

Antes do treinamento, esta etapa verifica:
- se as colunas obrigatórias existem
- se há valores nulos
- se o dataset foi carregado corretamente

Essa checagem evita falhas durante o fine-tuning.

In [ ]:
df_teste = df_medquad.sample(min(20, len(df_medquad)), random_state=42).copy()
print(df_teste.shape)

(20, 3)


## Teste inicial com subconjunto reduzido

Nesta etapa, é selecionada uma pequena amostra do dataset para validar se todo o pipeline funciona corretamente.

Esse teste é útil para:
- confirmar que o código está funcionando
- verificar se o modelo carrega corretamente
- identificar erros antes de rodar o treinamento completo

In [ ]:
result = run_medical_finetuning(
    df_medquad=df_teste,
    base_model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    output_dir="/content/drive/MyDrive/tech-challenge/artifacts/tinyllama-medquad-lora-test",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    max_seq_length=128,
    use_4bit=True,
    save_merged_model=False,
)

print(result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,No log,1.480533


{'status': 'success', 'base_model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'output_dir': '/content/drive/MyDrive/tech-challenge/artifacts/tinyllama-medquad-lora-test', 'train_samples': 18, 'val_samples': 2}


## Execução do fine-tuning

Esta célula executa o treinamento do modelo usando LoRA e quantização em 4 bits.

Parâmetros principais:
- `base_model`: modelo base utilizado
- `output_dir`: diretório onde os artefatos serão salvos
- `num_train_epochs`: número de épocas
- `gradient_accumulation_steps`: acumulação de gradiente para reduzir uso de memória
- `max_seq_length`: tamanho máximo das sequências

Ao final, os artefatos do adapter treinado são salvos no Google Drive.

In [ ]:
!ls -lah /content/drive/MyDrive/tech-challenge/artifacts/tinyllama-medquad-lora-test

total 28M
-rw------- 1 root root 1.1K Mar 22 05:42 adapter_config.json
-rw------- 1 root root  25M Mar 22 05:42 adapter_model.safetensors
-rw------- 1 root root  410 Mar 22 05:42 chat_template.jinja
drwx------ 2 root root 4.0K Mar 22 05:42 checkpoint-9
-rw------- 1 root root 5.1K Mar 22 05:42 README.md
-rw------- 1 root root  369 Mar 22 05:42 tokenizer_config.json
-rw------- 1 root root 3.5M Mar 22 05:42 tokenizer.json


## Verificação dos artefatos gerados

Após o treinamento, esta etapa lista os arquivos salvos no diretório de saída.

Os principais artefatos esperados são:
- `adapter_model.safetensors`
- `adapter_config.json`
- arquivos do tokenizer
- checkpoints intermediários

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "/content/drive/MyDrive/tech-challenge/artifacts/tinyllama-medquad-lora-test"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(base, adapter_path)
model.eval()

prompt = """### Instrução:
Responda à pergunta médica de forma clara, objetiva, cautelosa e informativa, sem prescrever tratamento definitivo nem substituir avaliação médica profissional.

### Pergunta:
Quais são os sintomas mais comuns do câncer de mama?

### Resposta:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instrução:
Responda à pergunta médica de forma clara, objetiva, cautelosa e informativa, sem prescrever tratamento definitivo nem substituir avaliação médica profissional.

### Pergunta:
Quais são os sintomas mais comuns do câncer de mama?

### Resposta:
A maioria dos casos de câncer de mama apresentam sintomas únicos ou complicados, como:
- Palpitações na região da testículo
- Presença de táxo (carciófago) em dias 1 a 5
- Fevereiro, março, abril, maio, junho, julho, agosto, setembro, outubro, novembro e dezembro
- Amanhã após uma fome
- Amanhã após alguma saída de águ


## Teste de inferência

Nesta etapa, o modelo base é recarregado e o adapter LoRA treinado é aplicado para gerar respostas a novas perguntas médicas.

O objetivo é verificar se o fine-tuning produziu um comportamento coerente com o domínio treinado.

## Interpretação dos resultados

Os resultados obtidos nesta fase devem ser analisados com cautela.

Como o teste inicial foi feito com uma amostra pequena do dataset, o modelo ainda pode gerar respostas incoerentes ou pouco confiáveis.

Esse comportamento é esperado em uma primeira validação técnica, cujo foco é verificar se o pipeline de fine-tuning está operacional.

In [ ]:
result_full = run_medical_finetuning(
    df_medquad=df_medquad.sample(min(500, len(df_medquad)), random_state=42),
    base_model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    output_dir="/content/drive/MyDrive/tech-challenge/artifacts/tinyllama-medquad-lora-v2",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    max_seq_length=256,
    use_4bit=True,
    save_merged_model=False,
)

print(result_full)

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.561048,0.415607
2,0.415787,0.377734


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'status': 'success', 'base_model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'output_dir': '/content/drive/MyDrive/tech-challenge/artifacts/tinyllama-medquad-lora-v2', 'train_samples': 450, 'val_samples': 50}


O modelo apresentou convergência de loss ao longo das épocas, indicando aprendizado dos padrões do dataset.

In [ ]:
result_full = run_medical_finetuning(
    df_medquad=df_medquad.sample(min(500, len(df_medquad)), random_state=42),
    base_model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    output_dir="/content/drive/MyDrive/tech-challenge/artifacts/tinyllama-medquad-lora-v3",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    max_seq_length=256,
    use_4bit=True,
    save_merged_model=False,
)

print(result_full)

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.553955,0.410923
2,0.400700,0.367854
3,0.229383,0.369977
4,0.226135,0.384031
5,0.133386,0.424734


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

{'status': 'success', 'base_model': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0', 'output_dir': '/content/drive/MyDrive/tech-challenge/artifacts/tinyllama-medquad-lora-v3', 'train_samples': 450, 'val_samples': 50}


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
adapter_path = "/content/drive/MyDrive/tech-challenge/artifacts/tinyllama-medquad-lora-test"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(base, adapter_path)
model.eval()

prompt = """### Instrução:
Responda à pergunta médica de forma clara, objetiva, cautelosa e informativa, sem prescrever tratamento definitivo nem substituir avaliação médica profissional.

Gere uma explicação em linguagem natural para um médico, baseada APENAS nos dados fornecidos.

Regras:
- Não invente informações (ex.: exames, sintomas, histórico) que não estejam nos dados.
- Se faltarem dados clínicos, diga explicitamente o que falta.
- Priorize sensibilidade (redução de falsos negativos).
- Tom profissional, objetivo, sem sensacionalismo.
- Inclua o aviso: "Não substitui avaliação médica".

### Pergunta:
Quais são os sintomas mais comuns do câncer de mama?

### Resposta:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instrução:
Responda à pergunta médica de forma clara, objetiva, cautelosa e informativa, sem prescrever tratamento definitivo nem substituir avaliação médica profissional.

Você é um assistente clínico para apoio ao diagnóstico de câncer de mama.
Gere uma explicação em linguagem natural para um médico, baseada APENAS nos dados fornecidos.

Regras:
- Não invente informações (ex.: exames, sintomas, histórico) que não estejam nos dados.
- Se faltarem dados clínicos, diga explicitamente o que falta.
- Priorize sensibilidade (redução de falsos negativos).
- Tom profissional, objetivo, sem sensacionalismo.
- Inclua o aviso: "Não substitui avaliação médica".

### Pergunta:
Quais são os sintomas mais comuns do câncer de mama?

### Resposta:
Os sintomas mais comuns do câncer de mama são:

1. Obstrução dos vômitos ou febre
2. Tachorreia
3. Nausea
4. Fevereiro
5. Queda de volume sanguíneo
6. Palidez
7. Edematose
8. Depressão
9. Dificuldade para se levantar
10. Sensibilidade à luz
11. Ataxia
1

## Análise dos resultados

Observa-se que a loss de treinamento diminui progressivamente ao longo das épocas, indicando que o modelo foi capaz de aprender padrões presentes no dataset.

A loss de validação, por sua vez, apresenta melhora até aproximadamente a segunda época, atingindo seu melhor valor nesse ponto. A partir da terceira época, verifica-se tendência de aumento da loss de validação, mesmo com a continuidade da redução da loss de treino.

Esse comportamento sugere início de overfitting, isto é, o modelo continua se ajustando cada vez mais ao conjunto de treinamento, mas perde parte de sua capacidade de generalização para dados não vistos.

Porém, ao analisarmos as respostas geradas, podemos identificar que o modelo não seguiu corretamente as informações existentes no dataset utilizado. Uma vez que elas estão bem aleatória indicando que o modelo alucionou e sem correlação aparente com a pergunta realizada.

Portanto, no próximo bloco, utilizaremos outro modelo base para verificar se hárespostas mais coerentes.

In [19]:
FINE_TUNING_PATH = "/content/repo_fiap/src/llm/fine_tuning.py"

print("Arquivo existe?", os.path.exists(FINE_TUNING_PATH))
if not os.path.exists(FINE_TUNING_PATH):
    raise FileNotFoundError(FINE_TUNING_PATH)

MODULE_NAME = "fiap_project_fine_tuning"

spec = importlib.util.spec_from_file_location(MODULE_NAME, FINE_TUNING_PATH)
mod = importlib.util.module_from_spec(spec)
sys.modules[MODULE_NAME] = mod

try:
    spec.loader.exec_module(mod)
    print("Import do arquivo OK")
except Exception:
    traceback.print_exc()
    raise

run_medical_finetuning = mod.run_medical_finetuning
prepare_finetuning_dataset = mod.prepare_finetuning_dataset
split_finetuning_dataset = mod.split_finetuning_dataset

Arquivo existe? True
Import do arquivo OK


In [20]:
df_medquad = df_medquad.drop_duplicates(subset=["question", "answer"]).copy()
df_medquad = df_medquad.dropna(subset=["question", "answer"]).copy()

df_medquad["question"] = df_medquad["question"].astype(str).str.strip()
df_medquad["answer"] = df_medquad["answer"].astype(str).str.strip()

df_medquad = df_medquad[
    (df_medquad["question"].str.len() > 10) &
    (df_medquad["answer"].str.len() > 20)
].copy()

print(df_medquad.shape)

(729, 3)


In [21]:
result_qwen = run_medical_finetuning(
    df_medquad=df_medquad.sample(min(500, len(df_medquad)), random_state=42),
    base_model="Qwen/Qwen2.5-1.5B-Instruct",
    output_dir="/content/drive/MyDrive/tech-challenge/artifacts/qwen25-1_5b-medquad-lora",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    max_seq_length=256,
    use_4bit=True,
    save_merged_model=False,
)

print(result_qwen)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
1,0.623293,0.424987
2,0.452656,0.394279


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


{'status': 'success', 'base_model': 'Qwen/Qwen2.5-1.5B-Instruct', 'output_dir': '/content/drive/MyDrive/tech-challenge/artifacts/qwen25-1_5b-medquad-lora', 'train_samples': 450, 'val_samples': 50}


In [22]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/tech-challenge/artifacts/qwen25-1_5b-medquad-lora"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model = PeftModel.from_pretrained(base, adapter_path)
model.eval()

prompt = """Você é um assistente para responder perguntas médicas de forma clara, objetiva e cautelosa.
Não invente informações.
Inclua ao final: "Não substitui avaliação médica".

Pergunta: Quais são os sintomas mais comuns do câncer de mama?
Resposta:"""

inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,  # importante!
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Você é um assistente para responder perguntas médicas de forma clara, objetiva e cautelosa.
Não invente informações.
Inclua ao final: "Não substitui avaliação médica".

Pergunta: Quais são os sintomas mais comuns do câncer de mama?
Resposta: The most common sign of breast cancer is a lump or thickening in the breast. Other signs and symptoms may include any of the following:         - A change in the size or shape of one side of the breast     - A dimple-like depression on the surface of the skin      - Inverted nipple (turned inward)       - Redness, scaliness, or thickened area on the skin of the areola (the dark area around the nipple)        These changes can be caused by many things other than breast cancer. It's important to talk with your doctor if you have these problems so


## Observações sobre o fine-tuning
- A troca do modelo base (TinyLlama → Qwen2.5-1.5B-Instruct) resultou em melhora significativa na qualidade das respostas.
- O modelo passou a gerar respostas coerentes, semanticamente corretas e mais alinhadas com conhecimento médico básico.
- A remoção de sampling (do_sample=False) foi essencial para reduzir alucinações.
- Modelos pequenos são altamente sensíveis à forma do prompt e à consistência do dataset.
## Limitações atuais
- O modelo ainda depende fortemente da qualidade dos dados de treino.
- Pode apresentar alucinações em perguntas mais complexas ou fora da distribuição.
- Não substitui validação médica real.
## Boas práticas adotadas
- Uso de geração determinística (sem sampling)
- Redução de temperatura e diversidade
- Prompt mais simples e direto
- Dataset com estrutura consistente

In [28]:
import os
print(os.getcwd())

/content/repo_fiap


In [30]:
from google.colab import drive
from pathlib import Path
import os

# remonta o Drive para garantir
drive.mount('/content/drive', force_remount=True)

# caminho final
output_path_json = Path(
    "/content/drive/MyDrive/tech-challenge/data/medical_qa/processed/medquad_cancergov_qa_finetuning.json"
)

# cria a pasta se não existir
output_path_json.parent.mkdir(parents=True, exist_ok=True)

# checagens
print("DataFrame vazio?", df_medquad.empty)
print("Quantidade de linhas:", len(df_medquad))
print("Diretório atual:", os.getcwd())
print("Pasta destino existe?", output_path_json.parent.exists())
print("Arquivo destino:", output_path_json)

# salva
df_medquad.to_json(output_path_json, orient="records", force_ascii=False, indent=2)

# confirma
print("Arquivo existe após salvar?", output_path_json.exists())

if output_path_json.exists():
    print("Tamanho em bytes:", output_path_json.stat().st_size)

# lista o conteúdo da pasta
print("\nArquivos na pasta:")
for f in output_path_json.parent.iterdir():
    print("-", f.name)

Mounted at /content/drive
DataFrame vazio? False
Quantidade de linhas: 729
Diretório atual: /content/repo_fiap
Pasta destino existe? True
Arquivo destino: /content/drive/MyDrive/tech-challenge/data/medical_qa/processed/medquad_cancergov_qa_finetuning.json
Arquivo existe após salvar? True
Tamanho em bytes: 5977484

Arquivos na pasta:
- medquad_cancergov_qa_finetuning.json


In [31]:
import pandas as pd

df_check = pd.read_json("/content/drive/MyDrive/tech-challenge/data/medical_qa/processed/medquad_cancergov_qa_finetuning.json")
print(df_check.head())
print(df_check.shape)

                                            question  \
0  What is (are) Adult Acute Lymphoblastic Leukem...   
1  What are the symptoms of Adult Acute Lymphobla...   
2  How to diagnose Adult Acute Lymphoblastic Leuk...   
3  What is the outlook for Adult Acute Lymphoblas...   
4  Who is at risk for Adult Acute Lymphoblastic L...   

                                              answer  \
0  Key Points\n                    - Adult acute ...   
1  Signs and symptoms of adult ALL include fever,...   
2  Tests that examine the blood and bone marrow a...   
3  Certain factors affect prognosis (chance of re...   
4  Previous chemotherapy and exposure to radiatio...   

                                                text  
0  ### Instrução:\nResponda à pergunta médica de ...  
1  ### Instrução:\nResponda à pergunta médica de ...  
2  ### Instrução:\nResponda à pergunta médica de ...  
3  ### Instrução:\nResponda à pergunta médica de ...  
4  ### Instrução:\nResponda à pergunta médica de ..